The code originally by Peter Norvig

https://github.com/norvig/pytudes/blob/main/ipynb/Sudoku.ipynb

Adapted for immutable boards, set[int], and other stuff


In [ ]:
import random

from board import Loc, Cell, Board, Digits, DIGITS

### Structure


In [ ]:
def allcells():
    return (Loc(r, c) for r in Loc.POS for c in Loc.POS)


def allunits():
    for b in Loc.POS:
        yield set(Loc(r, c) for r in Loc.row4box(b) for c in Loc.col4box(b))
    for r in Loc.POS:
        yield set(Loc(r, c) for c in Loc.POS)
    for c in Loc.POS:
        yield set(Loc(r, c) for r in Loc.POS)


def units(loc: Loc):
    b = Loc.box4loc(loc)
    return [
        tuple(Loc(r, c) for r in Loc.row4box(b) for c in Loc.col4box(b)),
        tuple(Loc(loc.r, c) for c in Loc.POS),
        tuple(Loc(r, loc.c) for r in Loc.POS),
    ]


def peers(loc: Loc):
    self = {loc}
    for unit in units(loc):
        yield from set(unit) - self


## Constraints


In [ ]:
def eliminate(board: Board, loc: Loc, dig: int) -> Board | None:
    """Eliminate d from grid[s]; implement the two constraint propagation strategies."""
    # print(f"{loc} -= {dig} ...")

    cell = board.get(loc)

    if dig not in cell:
        return board  ## Already eliminated

    cell = Cell(loc, Digits(cell.digits - {dig}))

    if not cell:
        # print(f"{loc} -= {dig} => void cell")
        return None  ## None: no legal digit left

    current = Board.insert(board, cell)

    # 1. If a square has only one possible digit, then eliminate that digit as a possibility for each of the square's peers.
    if len(cell) == 1:
        d2 = cell.final

        for l2 in peers(loc):
            current = eliminate(current, l2, d2)
            if current is None:
                # print(f"{loc} -= {dig} => {l2} -= {d2} failed")
                break  ## None: can't eliminate d2 from some square

    if current is None:
        return None

    for u in units(loc):
        occupied = [l for l in u if dig in current.get(l)]
        if not occupied:
            return None  ## None: no place in u for d
        if len(occupied) == 1:
            # 2. If a unit has only one possible square that can hold a digit, then fill the square with the digit.
            current = fill(current, occupied[0], dig)
        if current is None:
            # print(f"{loc} -= {dig} => {occupied[0]} += {dig} failed")
            break

    return current


def fill(board: Board, loc: Loc, dig: int) -> Board | None:
    """Eliminate all the digits except d from grid[s]."""
    # returns new board not modifying old
    # print(f"{loc} := {dig}")
    cell = board.get(loc)
    if cell.digits == {dig}:
        return board

    current = board
    for d2 in cell.digits - {dig}:
        current = eliminate(current, loc, d2)
        if current is None:
            break
    return current

In [ ]:
def constrain(board: Board) -> Board | None:
    "Propagate constraints on a copy of grid to yield a new constrained Grid."
    result = Board.Pristine()
    for cell in board:
        if len(cell) == 1:
            result = fill(result, cell.loc, cell.final)
            if result is None:
                break
    return result

## Search


In [ ]:
from collections.abc import Generator

from utils import draftboard


def search(current: Board | None) -> Board | None:
    "Depth-first search with constraint propagation to find a solution."
    if current is None:
        return None

    drafts = list(draftboard(current))
    if not len(drafts):
        return current

    drafts.sort(key=lambda c: len(c))
    target = drafts[0]  # minimum remaining values

    for d in target:  # random order
        solution = search(fill(current, target.loc, d))
        if solution:
            return solution
    return None


def search_all(current: Board | None) -> Generator[Board]:
    "Depth-first search with constraint propagation to find all solutions."
    if current is None:
        return

    drafts = list(draftboard(current))
    if not len(drafts):
        yield current
        return

    drafts.sort(key=lambda c: len(c))
    target = drafts[0]  # minimum remaining values

    for d in target:  # random order
        for solution in search_all(fill(current, target.loc, d)):
            yield solution

## Solving


In [ ]:
from utils import bparse, fillempty, picture

In [ ]:
def validate(solution: Board, puzzle: Board) -> bool:
    "Is this proposed solution to the puzzle actually valid?"
    return (
        all(len(cell) == 1 for cell in solution)
        and all(set(solution.get(l).final for l in unit) == DIGITS for unit in allunits())
        and all(solution.get(l).digits <= puzzle.get(l).digits for l in allcells())
    )

In [ ]:
def solve(puzzle: Board):
    print(picture(puzzle))
    solution = search(constrain(puzzle))
    assert solution is not None and validate(solution, puzzle)
    print("=================")
    print(picture(solution))

In [ ]:
def solve_count(puzzle: Board):
    print(picture(puzzle))
    found = 0
    for solution in search_all(constrain(puzzle)):
        assert validate(solution, puzzle)
        found += 1
    print(f"{found=}")

In [ ]:
puzzle = bparse("4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1.4......")  # 1 solution
# puzzle = bparse("4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1........") # 47688 solutions
puzzle = Board.transform(puzzle, fillempty)

In [ ]:
solve(puzzle)

In [ ]:
def check_uniq(puzzle: Board):
    found = []
    solving = search_all(constrain(puzzle))
    try:
        while len(found) < 2:
            solution = next(solving)
            found.append(solution)
            print("=================")
            print(picture(solution))
    except StopIteration:
        pass
    return len(found) == 1

## Generating


In [ ]:
# original function, using mutable values array

# def random_puzzle(N=17):
#     """Make a random puzzle with N or more assignments. Restart on contradictions.
#     Note the resulting puzzle is not guaranteed to be solvable, but empirically
#     about 99.8% of them are solvable. Some have multiple solutions."""
#     values = dict((s, digits) for s in squares)
#     for s in shuffled(squares):
#         if not assign(values, s, random.choice(values[s])):
#             break
#         ds = [values[s] for s in squares if len(values[s]) == 1]
#         if len(ds) >= N and len(set(ds)) >= 8:
#             return ''.join(values[s] if len(values[s])==1 else '.' for s in squares)
#     return random_puzzle(N) ## Give up and make a new puzzle

In [ ]:
def generate(N=17) -> Board | None:
    current = Board.Pristine()
    finals = []

    while len(finals) < N or len(set(finals)) < 8:
        cell = random.choice(tuple(draftboard(current)))
        dig = random.choice(tuple(cell.digits))
        current = fill(current, cell.loc, dig)
        if not current:
            break
        finals = [c.final for c in current if c.is_final]

    return current


In [ ]:
def generate_puzzle(n=17):
    attempts = 0
    while True:
        puzzle = generate(n)
        attempts += 1
        if puzzle:
            if attempts > 1:
                print(f"{attempts=}")
            return Board.transform(puzzle, lambda c: c if len(c) == 1 else Cell(c.loc, Digits(DIGITS)))

In [ ]:
puzzle = generate_puzzle(17)
print(picture(puzzle))

In [ ]:
check_uniq(puzzle)

## Examples


In [ ]:
# Arto Inkala
puzzle = bparse("""
..53.....
8......2.
.7..1.5..
4....53..
.1..7...6
..32...8.
.6.5....9
..4....3.
.....97..
""")

puzzle = Board.transform(puzzle, fillempty)

In [ ]:
solve(puzzle)

In [ ]:
check_uniq(puzzle)